In [ ]:
# Dependencies are managed by uv. From the repo root, run:
#     uv sync
# (No extra group needed for this notebook -- it uses the base dep set
# which already includes transformers, datasets, huggingface_hub,
# accelerate, scipy, scikit-learn, and tqdm.)
# Then launch: jupyter lab


In [ ]:
tinyaya_langs = ['amh', 'hau', 'ibo', 'mlg', 'sna', 'swh', 'wol', 'xho', 'yor', 'zul', 'tgl', 'msa', 'ind', 'vie', 'jav', 'khm', 'tha', 'lao', 'zho', 'mya', 'jpn', 'kor', 'hin', 'mar', 'ben', 'guj', 'pan', 'tam', 'tel', 'nep', 'ara', 'fas', 'urd', 'tur', 'mlt', 'heb', 'eng', 'nld', 'fra', 'ita', 'por', 'ron', 'spa', 'ces', 'pol', 'ukr', 'rus', 'ell', 'deu', 'dan', 'swe', 'nor', 'cat', 'glg', 'cym', 'gle', 'eus', 'hrv', 'lav', 'lit', 'slk', 'slv', 'est', 'fin', 'hun', 'srp', 'bul']

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoTokenizer
from collections import Counter
import numpy as np
import torch

# `tinyaya_langs` was undefined in the original Colab notebook. Use
# the canonical 67-ISO list from src/lid/constants.py instead.
from lid.constants import VALID_OPTIONS
tinyaya_langs = VALID_OPTIONS

load_dotenv()
HUGGING_FACE_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HF_NEW")
assert HUGGING_FACE_TOKEN, "Set HF_TOKEN (or HF_NEW) in your .env file at the repo root"
login(token=HUGGING_FACE_TOKEN)

dataset_name = "1024m/LID"
file_path = "Data_Main/LID-500.parquet"
full_dataset = load_dataset(
    "parquet",
    data_files={"train": f"hf://datasets/{dataset_name}/{file_path}"},
    token=HUGGING_FACE_TOKEN,
)["train"]
dataset = full_dataset.filter(lambda x: x["ISO-693-3"] in tinyaya_langs, num_proc=os.cpu_count())
tokenizer = AutoTokenizer.from_pretrained("CohereLabs/tiny-aya-global", token=HUGGING_FACE_TOKEN)


def tokenize_function(examples):
    return tokenizer(examples["text"], padding=False, truncation=True, max_length=128, add_special_tokens=False)


tokenized_dataset = dataset.map(tokenize_function, batched=True, num_proc=os.cpu_count(), remove_columns=["text"])
lang_stats: dict[str, Counter] = {}
iso_codes = dataset["ISO-693-3"]
input_ids = tokenized_dataset["input_ids"]
for lang, ids in zip(iso_codes, input_ids):
    if lang not in lang_stats:
        lang_stats[lang] = Counter()
    lang_stats[lang].update(ids)
model_weights = {
    lang: {int(t): float(np.log(c / sum(counts.values()))) for t, c in counts.items()}
    for lang, counts in lang_stats.items()
}
torch.save(model_weights, "tinyaya_lid_weights.pt")
print(f"Training Complete. Target Languages: {len(model_weights)} | Total Samples: {len(dataset)}")


In [ ]:
"""Alternate training variant (runs on LID-10000.parquet across all
languages). Not run by default. To use: replace Colab userdata with
the dotenv pattern from cell 2, then unwrap this docstring.

from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoTokenizer
from collections import Counter
import numpy as np
import torch
import os
HUGGING_FACE_TOKEN = userdata.get('HF_NEW')
login(token=HUGGING_FACE_TOKEN)
dataset_name = "1024m/LID"
file_path = "Data_Main/LID-10000.parquet"
dataset = load_dataset("parquet", data_files={"train": f"hf://datasets/{dataset_name}/{file_path}"}, token=HUGGING_FACE_TOKEN)["train"]
tokenizer = AutoTokenizer.from_pretrained("CohereLabs/tiny-aya-global", token=HUGGING_FACE_TOKEN)
def tokenize_function(examples):
    return tokenizer(examples["text"], padding=False, truncation=True, max_length=128, add_special_tokens=False)
tokenized_dataset = dataset.map(tokenize_function, batched=True, num_proc=os.cpu_count(), remove_columns=dataset.column_names)
lang_stats = {}
iso_codes = dataset["ISO-693-3"]
input_ids = tokenized_dataset["input_ids"]
for lang, ids in zip(iso_codes, input_ids):
    if lang not in lang_stats:
        lang_stats[lang] = Counter()
    lang_stats[lang].update(ids)
model_weights = {}
for lang, counts in lang_stats.items():
    total = sum(counts.values())
    model_weights[lang] = {t: np.log(c/total) for t, c in counts.items()}
torch.save(model_weights, "tinyaya_lid_weights.pt")
print(f"Dataset: {dataset_name} | Samples: {len(dataset)} | Languages: {len(model_weights)}")
print("Model training complete and saved to tinyaya_lid_weights.pt")
"""


In [ ]:
import pandas as pd
import json
import torch
import numpy as np
import os
from tqdm import tqdm
from scipy.special import softmax
from sklearn.metrics import f1_score
from datasets import load_dataset
from transformers import AutoTokenizer
weights = torch.load("tinyaya_lid_weights.pt", weights_only=False)
languages = list(weights.keys())
tokenizer = AutoTokenizer.from_pretrained("CohereLabs/tiny-aya-global")
vocab_size = tokenizer.vocab_size
lang_index = {lang: i for i, lang in enumerate(languages)}
weight_matrix = np.full((len(languages), vocab_size), -20.0)
for lang, counts in weights.items():
    for token_id, log_p in counts.items():
        if token_id < vocab_size:
            weight_matrix[lang_index[lang], token_id] = log_p
def run_inference(batch):
    tokenized = tokenizer(batch["text"], padding=False, truncation=True, max_length=128, add_special_tokens=False)["input_ids"]
    batch_probs_json = []
    batch_preds = []
    confidence_threshold = 0.10
    for ids in tokenized:
        if not ids:
            batch_preds.append("und")
            batch_probs_json.append("{}")
            continue
        scores = np.sum(weight_matrix[:, ids], axis=1)
        probs = softmax(scores)
        max_prob = np.max(probs)
        prob_dict = {languages[i]: f"{probs[i]:.10f}" for i in range(len(languages))}
        if max_prob < confidence_threshold:
            batch_preds.append("und")
        else:
            batch_preds.append(languages[np.argmax(scores)])
        batch_probs_json.append(json.dumps(prob_dict))
    return {"pred_lang": batch_preds, "probability_json": batch_probs_json}
benchmarks = [
    ("CommonLID", "Data_Benchmarks_Filtered/filtered_benchmark_CommonLID.parquet"),
    ("FLORES", "Data_Benchmarks_Filtered/filtered_benchmark_FLORES.parquet"),
    ("SmolSent", "Data_Benchmarks_Filtered/filtered_benchmark_SmolSent.parquet"),
    ("UDHRLID", "Data_Benchmarks_Filtered/filtered_benchmark_UDHRLID.parquet")
]
dataset_name = "1024m/LID"
all_benchmark_results = {}
summary_metrics = []
for name, path in benchmarks:
    ds = load_dataset("parquet", data_files={"test": f"hf://datasets/{dataset_name}/{path}"})["test"]
    res = ds.map(run_inference, batched=True, batch_size=2048, num_proc=os.cpu_count(), desc=f"Inference {name}")
    acc = (np.array(res["iso-693-3"]) == np.array(res["pred_lang"])).mean()
    f1 = f1_score(res["iso-693-3"], res["pred_lang"], average='macro')
    summary_metrics.append(f"{name} -> Accuracy: {acc:.4f}, Macro F1: {f1:.4f}")
    all_benchmark_results[name] = res
print("\nOVERALL METRICS:")
for line in summary_metrics:
    print(line)
for name, res in all_benchmark_results.items():
    print(f"\n{'='*10} {name} {'='*10}")
    df = pd.DataFrame({"true": res["iso-693-3"], "pred": res["pred_lang"]})
    for lang in sorted(df["true"].unique()):
        lang_acc = (df[df["true"] == lang]["true"] == df[df["true"] == lang]["pred"]).mean()
        print(f"{lang}: {lang_acc:.4f}")

In [ ]:
"""Alternate inference variant (kept for reference). Not run by
default. To use: unwrap this docstring.

import pandas as pd
import json
import torch
import numpy as np
import os
from tqdm import tqdm
from scipy.special import softmax
from sklearn.metrics import f1_score
from datasets import load_dataset
from transformers import AutoTokenizer
weights = torch.load("tinyaya_lid_weights.pt", weights_only=False)
languages = list(weights.keys())
tokenizer = AutoTokenizer.from_pretrained("CohereLabs/tiny-aya-global")
vocab_size = tokenizer.vocab_size
lang_index = {lang: i for i, lang in enumerate(languages)}
weight_matrix = np.full((len(languages), vocab_size), -20.0)
for lang, counts in weights.items():
    for token_id, log_p in counts.items():
        if token_id < vocab_size:
            weight_matrix[lang_index[lang], token_id] = log_p
def run_inference(batch):
    tokenized = tokenizer(batch["text"], padding=False, truncation=True, max_length=128, add_special_tokens=False)["input_ids"]
    batch_probs_json = []
    batch_preds = []
    confidence_threshold = 0.10
    for ids in tokenized:
        if not ids:
            batch_preds.append("und")
            batch_probs_json.append("{}")
            continue
        scores = np.sum(weight_matrix[:, ids], axis=1)
        probs = softmax(scores)
        max_prob = np.max(probs)
        prob_dict = {languages[i]: f"{probs[i]:.10f}" for i in range(len(languages))}
        if max_prob < confidence_threshold:
            batch_preds.append("und")
        else:
            batch_preds.append(languages[np.argmax(scores)])
        batch_probs_json.append(json.dumps(prob_dict))
    return {"pred_lang": batch_preds, "probability_json": batch_probs_json}
benchmarks = [
    ("CommonLID", "Data_Benchmarks_Filtered/filtered_benchmark_CommonLID.parquet"),
    ("FLORES", "Data_Benchmarks_Filtered/filtered_benchmark_FLORES.parquet"),
    ("SmolSent", "Data_Benchmarks_Filtered/filtered_benchmark_SmolSent.parquet"),
    ("UDHRLID", "Data_Benchmarks_Filtered/filtered_benchmark_UDHRLID.parquet")
]
dataset_name = "1024m/LID"
all_benchmark_results = {}
summary_metrics = []
for name, path in benchmarks:
    ds = load_dataset("parquet", data_files={"test": f"hf://datasets/{dataset_name}/{path}"})["test"]
    res = ds.map(run_inference, batched=True, batch_size=2048, num_proc=os.cpu_count(), desc=f"Inference {name}")
    acc = (np.array(res["iso-693-3"]) == np.array(res["pred_lang"])).mean()
    f1 = f1_score(res["iso-693-3"], res["pred_lang"], average='macro')
    summary_metrics.append(f"{name} -> Accuracy: {acc:.4f}, Macro F1: {f1:.4f}")
    all_benchmark_results[name] = res
print("\nOVERALL METRICS:")
for line in summary_metrics:
    print(line)
for name, res in all_benchmark_results.items():
    print(f"\n{'='*10} {name} {'='*10}")
    df = pd.DataFrame({"true": res["iso-693-3"], "pred": res["pred_lang"]})
    for lang in sorted(df["true"].unique()):
        lang_acc = (df[df["true"] == lang]["true"] == df[df["true"] == lang]["pred"]).mean()
        print(f"{lang}: {lang_acc:.4f}")
"""
